# Dataset setup and on-demand downloads

This notebook covers the setup side of `tnsd_access`: initialising a local dataset folder, inspecting the metadata, and seeing how on-demand zarr-store downloads work.

The epoch-loading examples assume this setup has already been done.

In [ ]:
from pathlib import Path

import pandas as pd

from tnsd_access import TrialHandler, init_dataset
from tnsd_access.utilities import check_islocal

pd.set_option("display.max_columns", 50)

## 1. Initialise the lightweight dataset files

`init_dataset(...)` creates the local dataset folder if needed and downloads the seed files: dataset-level files, session tables, manifests, and datastore metadata. It does not download all EEG zarr stores up front.

You need AWS credentials configured for the TNSD S3 bucket before running this.

In [ ]:
DATA_ROOT = Path("temporal-natural-scenes-dataset")
VERSION = "v1"

# Run this once, or whenever you want to refresh the lightweight metadata files.
RUN_INIT_DATASET = False

if RUN_INIT_DATASET:
    init_dataset(DATA_ROOT)

## 2. Open the metadata

`TrialHandler` reads `derivatives/<VERSION>/datastore/metadata.tsv` and resolves the zarr-store paths. This step should be fast because it only reads metadata.

In [ ]:
loader = TrialHandler(DATA_ROOT, version=VERSION)

print("metadata shape:", loader.metadata.shape)
loader.metadata.head()

In [ ]:
loader.metadata.columns.tolist()

A few quick summaries help confirm that the metadata loaded as expected.

In [ ]:
summary = loader.metadata.groupby("subject").agg(
    n_trials=("condition", "size"),
    n_conditions=("condition", "nunique"),
    n_sessions=("session", "nunique"),
)
summary.head(10)

In [ ]:
loader.metadata["shared"].value_counts(dropna=False).rename("n_trials")

## 3. Preview an on-demand download

The loader downloads zarr stores by store, not by individual row. This example looks at all shared-image trials for sub-01 and checks which underlying stores are already local.

In [ ]:
SUBJECT = 1

sub01_shared = loader.metadata.query("subject == @SUBJECT and shared == True").reset_index(drop=True)
store_paths = sub01_shared["path"].drop_duplicates().to_list()
local_status = check_islocal(store_paths)

print("sub-01 shared trial rows:", len(sub01_shared))
print("zarr stores needed:", len(store_paths))
print("stores present locally:", sum(local_status.values()))
print("stores missing locally:", len(store_paths) - sum(local_status.values()))

sub01_shared[["subject", "session", "run", "epoch", "condition", "shared", "path"]].head()

In [ ]:
missing_stores = [path for path, is_local in local_status.items() if not is_local]
missing_stores[:5]

## 4. Trigger the download through the loader

Calling `lookup_trials(subject=1, shared=True)` performs the same local-store check. If any stores are missing, it prompts you to download them from S3 and asks how many worker threads to use.

After the download, the returned trial table can be passed directly to `get_data(...)` in the epoch-loading notebook.

In [ ]:
DOWNLOAD_SUB01_SHARED = False

if DOWNLOAD_SUB01_SHARED:
    sub01_shared_trials = loader.lookup_trials(subject=1, shared=True)
else:
    sub01_shared_trials = sub01_shared

print("trial rows available:", len(sub01_shared_trials))
sub01_shared_trials.head()

If you set `DOWNLOAD_SUB01_SHARED = True`, answer `y` at the download prompt and choose a worker count such as `4` or `8`. The data are written under `DATA_ROOT / derivatives / VERSION / datastore`, and future lookups reuse the local stores.